# Model Experiments — Sensor Fault Detection

Comparing classifiers on the UCI SECOM dataset

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

df = pd.read_csv("wafer_secom_cleaned.csv")
X = df.drop(columns=["Good/Bad"])
y = np.where(df["Good/Bad"] == -1, 0, 1)  # 0 = Good, 1 = Faulty

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []  # every model's summary row goes here

In [2]:
def evaluate_model(name, model, X=X, y=y, cv=cv):
    """Runs 5-fold stratified CV, prints a report, stores the summary in `results`."""
    recalls, precisions, f1s = [], [], []
    tn = fp = fn = tp = 0

    for train_idx, test_idx in cv.split(X, y):
        model.fit(X.iloc[train_idx], y[train_idx])
        preds = model.predict(X.iloc[test_idx])
        y_true = y[test_idx]

        recalls.append(recall_score(y_true, preds, zero_division=0))
        precisions.append(precision_score(y_true, preds, zero_division=0))
        f1s.append(f1_score(y_true, preds, zero_division=0))
        cm = confusion_matrix(y_true, preds, labels=[0, 1])
        tn += cm[0, 0]; fp += cm[0, 1]; fn += cm[1, 0]; tp += cm[1, 1]

    print(f"{name}")
    print(f"  Recall:    {np.mean(recalls):.3f} ± {np.std(recalls):.3f}")
    print(f"  Precision: {np.mean(precisions):.3f} ± {np.std(precisions):.3f}")
    print(f"  F1:        {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"  Faults caught: {tp}/{tp+fn}   False alarms: {fp}/{fp+tn}")

    results.append({
        "model": name, "recall": np.mean(recalls), "recall_std": np.std(recalls),
        "precision": np.mean(precisions), "f1": np.mean(f1s),
        "faults_caught": f"{tp}/{tp+fn}", "false_alarms": f"{fp}/{fp+tn}",
    })